In [9]:
import numpy as np

from pprint import pprint
from pyvis.network import Network

import sys

sys.path.append('../src')

from config.config import PATHS

from patterns.weighted_alternations import WeightedPatternGenerator
from graphs.graph_creator import StrategyGraphBuilder
from graphs.cycle_analyzer import CycleAnalyzer
from graphs.graph_display import GraphVisualizer
from analysis.av_entropy import EntropyAnalyzer

### Initialize parameters

In [10]:
procs = {0:3, 1:3, 2:2, 3:2, 4:2, 5:2, 6:1, 7:1}
s = 4
Numpat = 1
permute_columns = True # permute agents
permute_rows = False # permute time
rng = np.random.default_rng(42)

### Generate the alternation pattern

In [11]:
gen = WeightedPatternGenerator(
    procs=procs,
    spots=s,
    num_patterns=Numpat,
    permute_columns=permute_columns,
    permute_rows=permute_rows,
    rng=rng
)

gen.generate_many()

pattern_file = PATHS['patterns'] / 'patterns.json'
gen.save(pattern_file)

### Get the agent strategies and dependencies' graphs

In [13]:
gb = StrategyGraphBuilder(rng)

gb.build_graphs()

### Get the average entropy
The average input information per node refers to the average number of neighbors, that is, the average number of bits received by the agents, which assumes that all possible strings are equiprobable.

The average entropy refers to the average number of bits used by agents computed with the actual frequency distribution of strings per pattern period. 

The average entropy is less than or equal to the average input info. This means that not all strings are equally represented in the input space, but sometimes there is an artifact when the period of the pattern is odd. 

In [14]:
ea = EntropyAnalyzer()

N = sum(list(procs.values()))
ea.compute_entropy_info(N,s,verbose=True)

Pattern 0: Average input information (bandwidth) per node: 1.00
Pattern 0: Average entropy of inputs per node: 0.81


([1.0], [np.float64(0.8112781244591326)])

### Look for the cycles in the dependencies' graphs

In [15]:
ca = CycleAnalyzer(
    num_nodes=N,
    num_steps=s,
)

ca.process()

[{'0a': {'pattern': ['1', '0', '0', '0'], 'neigh': ['4b'], 'strat': {'0': '0', '1': '1'}, 'input freq': {'0': 3, '1': 1}}, '0b': {'pattern': ['0', '0', '1', '0'], 'neigh': ['1a'], 'strat': {'0': '0', '1': '1'}, 'input freq': {'0': 3, '1': 1}}, '0c': {'pattern': ['0', '0', '0', '1'], 'neigh': ['7b'], 'strat': {'0': '0', '1': '1'}, 'input freq': {'0': 3, '1': 1}}, '1a': {'pattern': ['0', '1', '0', '0'], 'neigh': ['7a'], 'strat': {'1': '1', '0': '0'}, 'input freq': {'1': 1, '0': 3}}, '2a': {'pattern': ['1', '0', '0', '0'], 'neigh': ['6b'], 'strat': {'0': '0', '1': '1'}, 'input freq': {'0': 3, '1': 1}}, '2b': {'pattern': ['0', '1', '0', '0'], 'neigh': ['3a'], 'strat': {'1': '1', '0': '0'}, 'input freq': {'1': 1, '0': 3}}, '3a': {'pattern': ['1', '0', '0', '0'], 'neigh': ['7c'], 'strat': {'0': '0', '1': '1'}, 'input freq': {'0': 3, '1': 1}}, '3b': {'pattern': ['0', '0', '1', '0'], 'neigh': ['2b'], 'strat': {'0': '0', '1': '1'}, 'input freq': {'0': 3, '1': 1}}, '4a': {'pattern': ['0', '1', '

### Visualize the dependencies' graphs

In [16]:
gv = GraphVisualizer(
    num_nodes=N,
    num_steps=s,
    mode='id'
)

nets, output_files = gv.generate_all_patterns(width="600px", height="600px", physics=True)

for i in range(len(nets)):
    nets[i].show(str(output_files[i]), notebook=False)

/home/carlos/Documents/Information/edgar_german/repo/alternation_EFP/data/html/graph_N16s4_pattern0.html
